In [ ]:
import pandas as pd
import numpy as np

# ===============================
# LOAD FILE
# ===============================

file_path = "production_17_feb.xlsx"   # change to your file
df = pd.read_excel(file_path)

# Clean column names
df.columns = df.columns.str.strip()

# ===============================
# BASIC CALCULATIONS
# ===============================

# Reject %
df["Reject_%"] = np.where(
    df["prodn"] > 0,
    (df["rej"] / df["prodn"]) * 100,
    0
)

# Quality Yield
df["Yield_%"] = np.where(
    df["prodn"] > 0,
    (df["ok"] / df["prodn"]) * 100,
    0
)

# Cycle deviation %
df["Cycle_Deviation_%"] = np.where(
    df["cycle time tgt"] > 0,
    ((df["cycle time act"] - df["cycle time tgt"]) / df["cycle time tgt"]) * 100,
    0
)

# Cavity utilization %
df["Cavity_Utilization_%"] = np.where(
    df["planned cavity"] > 0,
    (df["actual cavity"] / df["planned cavity"]) * 100,
    0
)

# Availability indicator
df["Total_Loss_Time"] = (
    df["downtime"] +
    df["co time"] +
    df["break time"]
)

# ===============================
# SUMMARY VIEW
# ===============================

summary_cols = [
    "part no",
    "ok",
    "rej",
    "prodn",
    "Reject_%",
    "Yield_%",
    "Cycle_Deviation_%",
    "Cavity_Utilization_%",
    "Total_Loss_Time",
    "pe"
]

summary_df = df[summary_cols]

# ===============================
# SAVE
# ===============================

summary_df.to_excel("Production_17Feb_Diagnostic.xlsx", index=False)

print("Production diagnostic completed.")


In [ ]:
import pandas as pd
import numpy as np

# ===============================
# LOAD FILE
# ===============================

file_path = "production_17_feb.xlsx"   # change path if needed
df = pd.read_excel(file_path)

# Clean column names
df.columns = df.columns.str.strip().str.lower()

# ===============================
# CHECK REQUIRED COLUMNS
# ===============================

required_cols = [
    "part no",
    "prodn",
    "run time",
    "cycle time act",
    "actual cavity",
    "pdt time"
]

missing = [col for col in required_cols if col not in df.columns]

if missing:
    print("Missing columns:", missing)
    print("Check column names in file.")
else:

    print("\n===== STEP 1: CAVITY TEST =====")

    # Theoretical production assuming cavity = running cavities
    df["expected_prodn"] = (
        (df["run time"] / df["cycle time act"]) * df["actual cavity"]
    )

    df["prodn_diff"] = df["prodn"] - df["expected_prodn"]

    print(df[["part no", "prodn", "expected_prodn", "prodn_diff"]].head(10))

    avg_diff = df["prodn_diff"].abs().mean()
    print("\nAverage production difference:", avg_diff)

    if avg_diff < df["prodn"].mean() * 0.1:
        print("👉 Actual cavity likely represents running cavities.")
    else:
        print("👉 Actual cavity likely represents tool capacity or something else.")

    print("\n===== STEP 2: RUN TIME TEST =====")

    corr_runtime = df[["run time", "prodn"]].corr().iloc[0,1]
    print("Correlation between Run Time and Production:", corr_runtime)

    if corr_runtime > 0.7:
        print("👉 Run time likely represents actual machine operating time.")
    else:
        print("👉 Run time may not be pure operating time — needs clarification.")

    print("\n===== STEP 3: PDT TIME TEST =====")

    corr_pdt = df[["pdt time", "run time"]].corr().iloc[0,1]
    print("Correlation between PDT Time and Run Time:", corr_pdt)

    if corr_pdt < 0:
        print("👉 PDT likely represents planned downtime.")
    else:
        print("👉 PDT meaning unclear — may not directly reduce run time.")

# ===============================
# SAVE RESULTS
# ===============================

df.to_excel("production_definition_test_results.xlsx", index=False)

print("\nDiagnostic results saved.")


In [ ]:
import pandas as pd
import numpy as np

# ===============================
# LOAD FILE
# ===============================

file_path = "D:/farukhnagar plant part reports/Machine Part Report Plant 17 feb.xlsx"
df = pd.read_excel(file_path)

# Clean column names (remove extra spaces only)
df.columns = df.columns.str.strip()

# ===============================
# VERIFY REQUIRED COLUMNS
# ===============================

required_cols = [
    "Part Number",
    "Prodn",
    "Run Time",
    "CycleTime Act",
    "Actual Cavity",
    "Pdt Time"
]

missing = [col for col in required_cols if col not in df.columns]

if missing:
    print("❌ Missing columns:", missing)
    print("👉 Run this to inspect headers:")
    print(df.columns.tolist())

else:

    print("\n===== STEP 1: CAVITY TEST =====")

    # Theoretical production assuming cavity = running cavities
    df["Expected Prodn"] = (
        (df["Run Time"] / df["CycleTime Act"]) * df["Actual Cavity"]
    )

    df["Prodn Diff"] = df["Prodn"] - df["Expected Prodn"]

    print(df[[
        "Part Number",
        "Prodn",
        "Expected Prodn",
        "Prodn Diff"
    ]].head(10))

    avg_diff = df["Prodn Diff"].abs().mean()
    print("\nAverage production difference:", avg_diff)

    if avg_diff < df["Prodn"].mean() * 0.1:
        print("👉 Actual Cavity likely represents RUNNING cavities.")
    else:
        print("👉 Actual Cavity likely represents TOOL capacity or needs clarification.")

    print("\n===== STEP 2: RUN TIME TEST =====")

    corr_runtime = df[["Run Time", "Prodn"]].corr().iloc[0,1]
    print("Correlation between Run Time and Production:", corr_runtime)

    if corr_runtime > 0.7:
        print("👉 Run Time likely represents actual machine operating time.")
    else:
        print("👉 Run Time meaning unclear — may include other factors.")

    print("\n===== STEP 3: PDT TIME TEST =====")

    corr_pdt = df[["Pdt Time", "Run Time"]].corr().iloc[0,1]
    print("Correlation between PDT Time and Run Time:", corr_pdt)

    if corr_pdt < 0:
        print("👉 PDT Time likely represents planned downtime.")
    else:
        print("👉 PDT Time meaning unclear — may not directly reduce runtime.")

# ===============================
# SAVE RESULTS
# ===============================

df.to_excel("production_definition_test_results.xlsx", index=False)

print("\n✅ Diagnostic results saved to file.")
